# FalaPicto — Ajuste fino do YOLOv8n (transfer learning)

O YOLOv8n pré-treinado (COCO) não conhece lápis, apontador, caneta etc. e confunde garrafa com copo.
Aqui ajustamos o modelo com fotos próprias. **Os nomes das classes devem ser os ids dos cards** (`lapis`, `copo`, `garrafa`, `caderno`...),
assim a API usa direto (sem mapeamento COCO).

Pré-requisito: dataset no formato YOLOv8 (Roboflow → Export → *YOLOv8*), baixado como `.zip` (pastas train/valid/test + data.yaml).
Use GPU T4: Ambiente de execução > Alterar tipo.

In [ ]:
!pip install -q ultralytics

In [ ]:
# Envie o .zip do dataset exportado do Roboflow (formato YOLOv8)
from google.colab import files
up = files.upload()
nome = list(up.keys())[0]
!rm -rf ds && mkdir ds && unzip -q "{nome}" -d ds
!cat ds/data.yaml

In [ ]:
from ultralytics import YOLO
model = YOLO('yolov8n.pt')                       # parte do COCO (transfer learning)
model.train(data='ds/data.yaml', epochs=60, imgsz=640, batch=16, patience=15, seed=0,
            hsv_h=0.02, hsv_s=0.6, hsv_v=0.4, degrees=10, fliplr=0.5, mosaic=1.0)   # aumento de dados p/ poucas fotos

In [ ]:
# Métricas por classe no conjunto de validação (use na tabela do relatório)
m = YOLO('runs/detect/train/weights/best.pt')
r = m.val(data='ds/data.yaml', split='val')
print('mAP50 geral:', round(r.box.map50,3), '| mAP50-95:', round(r.box.map,3))
for i, c in enumerate(r.box.ap_class_indices):
    print(f'{m.names[int(c)]:12s} P={r.box.p[i]:.2f}  R={r.box.r[i]:.2f}  mAP50={r.box.ap50[i]:.2f}')

In [ ]:
# Guarda os pesos no Drive e baixa
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/falapicto_yolo
!cp runs/detect/train/weights/best.pt /content/drive/MyDrive/falapicto_yolo/falapicto_yolo.pt
!cp -r runs/detect/train /content/drive/MyDrive/falapicto_yolo/curvas   # curvas e matriz de confusão p/ o relatório
files.download('runs/detect/train/weights/best.pt')

## Depois
1. Renomeie `best.pt` para `falapicto_yolo.pt` e coloque em `falapicto/models/`.
2. `docker build -t falapicto .` e recrie o container (a API usa esse arquivo automaticamente).
3. Meça no PC: `python avaliar_yolo.py fotos --pesos ..\models\falapicto_yolo.pt --classes-sao-cards`